# Analyzing Data Science Task to LMs

In [1]:
# imports
import yaml
import json
from stat_genie.blade_pipeline.baselines.config import MultiRunConfig
from stat_genie.blade_pipeline.baselines.multirun import multirun_llm
from stat_genie.blade_pipeline.additions.prompt.prompt import PromptGenerator
import os
from os.path import join
from pathlib import Path
from stat_genie.blade_pipeline.additions.analysis.get_model_output import get_model_output
from stat_genie.blade_pipeline.additions.perturbations.feature_names import FeaturePerturbation
from stat_genie.blade_pipeline.additions.analysis.fix_code import check_and_fix_code
import importlib.util
import sys
import pandas as pd
from stat_genie.blade_pipeline.baselines.multirun import _format_cvars_for_prompt
from stat_genie.blade_pipeline.additions.analysis.conclusion import write_final_answer_code, make_conclusion

In [2]:
### set config parameters

# set up config object
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_config = yaml.safe_load(open("../../config/llm_config.yml"))
llm_config["provider"] = llm_provider
llm_config["model"] = llm_model
llm_eval_config = llm_config

# set rest of parameters
output_dir = "analysis1_output"
run_dataset = "caschools"
use_agent = False
use_data_desc = True
num_runs=3
use_code_cache=False

In [3]:
# the MultiRunConfig object is how BLADE standardizes experiment configuration
single_run_config = MultiRunConfig(llm=llm_config,
                llm_eval=llm_eval_config,
                output_dir=output_dir,
                run_dataset=run_dataset,
                use_agent=use_agent,
                use_data_desc=use_data_desc,
                num_runs=num_runs,
                use_code_cache=use_code_cache,
                fix_code=True,
)

In [4]:
# prevent cache use
single_run_config.llm.use_cache = False
single_run_config.llm_eval.use_cache = False

In [5]:
### run the experiment
# version #1 of the analysis does NOT apply any feature perturbation.
multirun_llm(single_run_config)

[2025-12-05 10:43:09.96][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-05 10:43:10.36][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-05 10:43:10.59][llm.py:109 - stat_genie.blade_pipeline.llms.llm:generate][PROMPT] Sending prompt from <class 'stat_genie.blade_pipeline.baselines.lm.gen_analysis.GenAnalysisLM'>
===================[[system]]===================
You are an AI Data Analysis Assistant who is an expert at writing an end-to-end scientific analysis given a research question and a dataset. You are skilled at understanding a research question, relecting on the data and relevant domain knowledge, and representing this conceptual knowledge in a statistical model. Key to this modeling process is formalizing the conceptual model, which inc